# Projet MLSF – Partie I : solveur SAT pour Meowdoku
**Pablo Bosch, Cécile Chevrière**

Les programmes sont regroupés dans `meowdoku.py` (grilles, encodage CNF, DIMACS, vérification) et `generateur.py` (génération de grilles). Ce notebook les exécute dans l'ordre des questions ; la démarche et l'analyse des résultats sont dans le rapport.

In [13]:
import itertools, os, random
from meowdoku import (lire_grille, ecrire_grille, afficher, grille_vers_cnf, ecrire_dimacs,
                      convertir, var, satisfait, chats_depuis_valuation, verifier_solution)
from generateur import generer_grille, generer_fichier

os.makedirs("grilles", exist_ok=True)
os.makedirs("cnf", exist_ok=True)

## Question 1 – Modélisation de Meowdoku

### 1.1 Conversion grille → DIMACS
Grille de la figure 1 de l'énoncé : lecture, conversion, aperçu du fichier produit.

In [14]:
grille = lire_grille("grilles/grille_enonce.txt")
afficher(grille)
nb_vars, nb_clauses = convertir("grilles/grille_enonce.txt", "cnf/grille_enonce.cnf")
print(f"\n{nb_vars} variables, {nb_clauses} clauses")
with open("cnf/grille_enonce.cnf") as f:
    print("".join(f.readlines()[:10]) + "...")

A B C B B D
A B B B B D
A A B D D D
A A A E D D
A A A A D D
F A D D D D

36 variables, 309 clauses
c Meowdoku : grilles/grille_enonce.txt
c GRID 6 6, 6 regions
c variable x_(i,j) = (i-1)*6 + j  (chat sur la case (i,j))
p cnf 36 309
-1 -2 0
-1 -3 0
-1 -4 0
-1 -5 0
-1 -6 0
-2 -3 0
...


### 1.2 Validation de l'encodage
**a.** La solution de la figure 2 est licite et satisfait la CNF.

In [15]:
solution_fig2 = {(3, 2), (2, 5), (1, 3), (5, 6), (4, 4), (6, 1)}
_, clauses = grille_vers_cnf(grille)
print("règles violées :", verifier_solution(grille, solution_fig2))
print("CNF satisfaite :", satisfait(clauses, {var(i, j, grille.M) for i, j in solution_fig2}))

règles violées : []
CNF satisfaite : True


**b.** Test exhaustif sur des grilles 4×4 : pour chacune des 2¹⁶ valuations, on compare « la valuation satisfait la CNF » et « la configuration de chats associée est licite ».

In [16]:
def comparer(grille, clauses, vraies):
    return satisfait(clauses, vraies) == (not verifier_solution(grille, chats_depuis_valuation(vraies, grille.M)))

for graine in range(5):
    g, _ = generer_grille(4, graine)
    nv, cl = grille_vers_cnf(g)
    modeles, accord = 0, True
    for bits in range(2 ** nv):
        vraies = {v for v in range(1, nv + 1) if bits >> (v - 1) & 1}
        accord &= comparer(g, cl, vraies)
        modeles += satisfait(cl, vraies)
    print(f"grille 4x4 n°{graine} : accord sur les 2^{nv} valuations = {accord}, modèles = {modeles}")

grille 4x4 n°0 : accord sur les 2^16 valuations = True, modèles = 1
grille 4x4 n°1 : accord sur les 2^16 valuations = True, modèles = 1
grille 4x4 n°2 : accord sur les 2^16 valuations = True, modèles = 2
grille 4x4 n°3 : accord sur les 2^16 valuations = True, modèles = 1
grille 4x4 n°4 : accord sur les 2^16 valuations = True, modèles = 2


### 1.3 Générateur de grilles (N = M = K)
Exemple de grille 8×8 générée, avec la solution construite (`*`).

In [17]:
g, chats = generer_fichier(8, 2026, "grilles/exemple_8x8.txt")
afficher(g, chats)
print("\nrègles violées par la solution construite :", verifier_solution(g, chats))

B B B E A A * A
* B E E A A A C
B B E E A C C *
B E E E * C C C
E E * D D D F F
G G G H F * F F
G * G H H F F F
G G H * H H F F

règles violées par la solution construite : []


Pas de configuration licite pour N = 2 et N = 3 :

In [18]:
for N in (2, 3):
    try:
        generer_grille(N)
    except ValueError as e:
        print(e)

aucune configuration licite pour N = 2 (impossible pour N = 2, 3)
aucune configuration licite pour N = 3 (impossible pour N = 2, 3)


### 1.4 Jeu de grilles générées et fichiers DIMACS
Grilles de taille 4 à 20 (3 graines par taille), vérification de la solution construite et conversion en DIMACS. Ces instances serviront aux questions suivantes.

In [19]:
print(f"{'N':>3} {'variables':>10} {'clauses':>9}  solutions construites licites")
for N in range(4, 21):
    tailles, licites = [], True
    for graine in range(3):
        nom = f"meow_{N:02d}_{graine}"
        g, chats = generer_fichier(N, graine, f"grilles/{nom}.txt")
        licites &= not verifier_solution(g, chats)
        tailles.append(convertir(f"grilles/{nom}.txt", f"cnf/{nom}.cnf"))
    nv = tailles[0][0]
    nc = sum(t[1] for t in tailles) // 3
    print(f"{N:>3} {nv:>10} {nc:>9}  {licites}")

  N  variables   clauses  solutions construites licites
  4         16        74  True
  5         25       146  True
  6         36       258  True
  7         49       412  True
  8         64       638  True
  9         81       909  True
 10        100      1253  True
 11        121      1671  True
 12        144      2207  True
 13        169      2807  True
 14        196      3616  True
 15        225      4341  True
 16        256      5423  True
 17        289      6568  True
 18        324      7720  True
 19        361      9039  True
 20        400     10547  True


## Question 2 – Solveur DPLL

La lecture DIMACS (`lire_dimacs`) et les algorithmes 1 et 2 (`dpll_statut`, `dpll`) sont dans `dpll.py`.

In [20]:
from dpll import lire_dimacs, dpll_statut, dpll, completer, verifier_modele

### Validation
**a.** Comparaison avec la force brute ($2^n$ valuations) sur 500 petites formules aléatoires.

In [21]:
rng = random.Random(0)
nb_sat = 0
for _ in range(500):
    n = rng.randint(1, 8)
    F = [[rng.choice((1, -1)) * rng.randint(1, n) for _ in range(rng.randint(1, 3))]
         for _ in range(rng.randint(0, 30))]
    ref = any(verifier_modele(F, [i + 1 if b else -(i + 1) for i, b in enumerate(bits)])
              for bits in itertools.product((0, 1), repeat=n))
    assert dpll_statut(F, rng) == ref
    sat, v = dpll(F, rng)
    assert sat == ref and (not sat or verifier_modele(F, completer(v, n)))
    nb_sat += sat
print(f"500 formules : accord avec la force brute ({nb_sat} SAT, {500 - nb_sat} UNSAT)")

500 formules : accord avec la force brute (154 SAT, 346 UNSAT)


**b.** Instances fournies par les enseignants (`../../public-instances/`), avec la limite de 60 s de la compétition. Chaque exécution passe par la ligne de commande `python dpll.py` dans un processus séparé, ce qui permet de l'interrompre ; les modèles renvoyés sont vérifiés.

In [22]:
import json, subprocess, sys, time
from concurrent.futures import ThreadPoolExecutor

LIMITE = 60
attendu = json.load(open("../../public-instances/expected.json"))

def executer(nom, algo, graine=0):
    chemin = f"../../public-instances/instances/{nom}"
    t0 = time.perf_counter()
    try:
        r = subprocess.run([sys.executable, "dpll.py", chemin, "--algo", str(algo),
                            "--graine", str(graine)], capture_output=True, text=True, timeout=LIMITE)
    except subprocess.TimeoutExpired:
        return "TIMEOUT", None, LIMITE
    duree = time.perf_counter() - t0
    sortie = r.stdout.splitlines()
    statut = "SAT" if "s SATISFIABLE" in sortie else "UNSAT"
    correct = statut == attendu[nom]
    if correct and statut == "SAT" and algo == 2:
        modele = [int(x) for x in sortie[1].split()[1:-1]]
        correct = verifier_modele(lire_dimacs(chemin)[1], modele)
    return statut, correct, duree

taches = [(nom, algo) for nom in sorted(attendu) for algo in (1, 2)]
with ThreadPoolExecutor(4) as ex:
    resultats = dict(zip(taches, ex.map(lambda t: executer(*t), taches)))

print(f"{'instance':<34}{'attendu':>8}   {'algo 1':>16}   {'algo 2':>16}")
for nom in sorted(attendu):
    cases = []
    for algo in (1, 2):
        statut, correct, duree = resultats[nom, algo]
        cases.append("timeout" if statut == "TIMEOUT"
                     else f"{statut} {'ok' if correct else 'FAUX'} {duree:5.2f}s")
    print(f"{nom:<34}{attendu[nom]:>8}   {cases[0]:>16}   {cases[1]:>16}")

instance                           attendu             algo 1             algo 2
cli_model_sat.cnf                      SAT      SAT ok  0.04s      SAT ok  0.04s
edge_all_negative_sat.cnf              SAT      SAT ok  0.04s      SAT ok  0.04s
edge_duplicate_literals_unsat.cnf    UNSAT    UNSAT ok  0.04s    UNSAT ok  0.04s
edge_empty_clause_unsat.cnf          UNSAT    UNSAT ok  0.04s    UNSAT ok  0.04s
edge_empty_formula_sat.cnf             SAT      SAT ok  0.04s      SAT ok  0.04s
edge_no_clause_sat.cnf                 SAT      SAT ok  0.04s      SAT ok  0.04s
edge_tautology_sat.cnf                 SAT      SAT ok  0.04s      SAT ok  0.04s
public-01.cnf                          SAT      SAT ok  0.04s      SAT ok  0.04s
public-02.cnf                          SAT      SAT ok  0.05s      SAT ok  0.06s
public-03.cnf                        UNSAT            timeout            timeout
public-04.cnf                          SAT      SAT ok  0.10s      SAT ok  0.09s
public-05.cnf               

**c.** Grilles Meowdoku de la question 1 : DPLL trouve une solution, contrôlée par `verifier_solution`.

In [23]:
for N in (4, 6, 8, 10, 12):
    g = lire_grille(f"grilles/meow_{N:02d}_0.txt")
    nv, cl = grille_vers_cnf(g)
    t0 = time.perf_counter()
    sat, v = dpll(cl, random.Random(0))
    duree = time.perf_counter() - t0
    chats = chats_depuis_valuation({p for p, b in v.items() if b}, g.M)
    print(f"{N:>2}x{N:<2} : SAT = {sat}, règles violées = {verifier_solution(g, chats)}, {duree:.3f} s")

 4x4  : SAT = True, règles violées = [], 0.000 s
 6x6  : SAT = True, règles violées = [], 0.001 s
 8x8  : SAT = True, règles violées = [], 0.004 s
10x10 : SAT = True, règles violées = [], 0.026 s
12x12 : SAT = True, règles violées = [], 0.042 s


## Question 3 – Résolution d'une grille Meowdoku

`resoudre.py` : grille → fichier DIMACS (question 1) → DPLL, algorithme 2 (question 2) → fichier solution au format de la figure 2. Le fichier écrit est relu et vérifié (`verifier_solution`).

**a.** Grille de la figure 1 de l'énoncé :

In [24]:
from resoudre import resoudre

os.makedirs("solutions", exist_ok=True)
resoudre("grilles/grille_enonce.txt", "solutions/grille_enonce.txt")
with open("solutions/grille_enonce.txt") as f:
    print(f.read())

c Grille 1
GRID 6 6
ZONES A B C D E F

A B C B B D
A B B B B D
A A B D D D
A A A E D D
A A A A D D
F A D D D D

A 3 2
B 2 5
C 1 3
D 5 6
E 4 4
F 6 1

. . * . . .
. . . . * .
. * . . . .
. . . * . .
. . . . . *
* . . . . .



**b.** Les 51 grilles générées à la question 1 (N = 4 à 20, 3 graines par taille), avec la limite de 60 s : chaque résolution passe par `python resoudre.py` dans un processus séparé, qui s'arrête avec une erreur si la solution écrite est illicite.

In [25]:
def resoudre_limite(nom):
    t0 = time.perf_counter()
    try:
        r = subprocess.run([sys.executable, "resoudre.py", f"grilles/{nom}.txt", f"solutions/{nom}.txt"],
                           capture_output=True, text=True, timeout=LIMITE)
    except subprocess.TimeoutExpired:
        return None
    if r.returncode != 0:
        raise RuntimeError(f"{nom} : {r.stderr}")
    return time.perf_counter() - t0

noms = [f"meow_{N:02d}_{graine}" for N in range(4, 21) for graine in range(3)]
with ThreadPoolExecutor(4) as ex:
    durees = dict(zip(noms, ex.map(resoudre_limite, noms)))

print(f"{'N':>3}   {'graine 0':>9} {'graine 1':>9} {'graine 2':>9}")
for N in range(4, 21):
    cases = [durees[f"meow_{N:02d}_{g}"] for g in range(3)]
    print(f"{N:>3}   " + " ".join("  timeout" if d is None else f"{d:8.2f}s" for d in cases))
print(f"\n{sum(d is not None for d in durees.values())}/51 grilles résolues en moins de {LIMITE} s, "
      "toutes les solutions écrites sont vérifiées")

  N    graine 0  graine 1  graine 2
  4       0.06s     0.06s     0.05s
  5       0.05s     0.04s     0.04s
  6       0.04s     0.04s     0.04s
  7       0.04s     0.04s     0.05s
  8       0.05s     0.04s     0.04s
  9       0.07s     0.05s     0.05s
 10       0.05s     0.06s     0.06s
 11       0.09s     0.06s     0.13s
 12       0.15s     0.15s     0.16s
 13       0.16s     4.75s     0.16s
 14       0.23s     0.25s     0.24s
 15       0.56s     0.29s     0.42s
 16      48.62s     2.99s     0.67s
 17       0.57s     1.49s     0.51s
 18      12.51s     0.67s    16.12s
 19      23.05s     1.24s   timeout
 20       0.99s     1.01s    57.93s

50/51 grilles résolues en moins de 60 s, toutes les solutions écrites sont vérifiées


## Question 4 – Heuristiques de branchement

`heuristiques.py` : algorithme 3 (`dpll_h`), identique à l'algorithme 2 sauf la ligne 17, où la décision (p, b) est donnée par une heuristique au lieu d'un tirage aléatoire. Quatre heuristiques, toutes recalculées sur la formule courante (versions dynamiques) :

| | Heuristique | Variable p | Valeur b |
|---|---|---|---|
| H1 | Première Variable Libre | plus petite variable de F | 1 |
| H2 | Variable la plus fréquente, polarité majoritaire | max #(p, F) | 1 si #⁺ ≥ #⁻ |
| H3 | Littéral le plus fréquent | littéral l maximisant occ(l, F) | 1 si l = p, 0 si l = ¬p |
| H4 | Variable la plus fréquente dans les clauses de taille minimale | max #ₖ(p, F), k = taille minimale | 1 si #ₖ⁺ ≥ #ₖ⁻ |

Égalités départagées par la plus petite variable. Choix de chaque heuristique sur la formule d'exemple de l'énoncé :

In [26]:
from heuristiques import HEURISTIQUES, dpll_h

# formule d'exemple de l'énoncé : (x1 ∨ x2 ∨ ¬x4) ∧ (x2 ∨ x3) ∧ (x1 ∨ ¬x2 ∨ x4 ∨ ¬x5)
F = [[1, 2, -4], [2, 3], [1, -2, 4, -5]]
for nom, h in HEURISTIQUES.items():
    p, b = h(F)
    print(f"{nom} : x{p} <- {int(b)}")

H1 : x1 <- 1
H2 : x2 <- 1
H3 : x1 <- 1
H4 : x2 <- 1


### Validation
Les heuristiques ne doivent changer que l'ordre d'exploration, jamais la réponse.

**a.** Comparaison avec la force brute sur 500 petites formules aléatoires, pour chaque heuristique.

In [27]:
rng = random.Random(1)
formules = []
for _ in range(500):
    n = rng.randint(1, 8)
    F = [[rng.choice((1, -1)) * rng.randint(1, n) for _ in range(rng.randint(1, 3))]
         for _ in range(rng.randint(0, 30))]
    ref = any(verifier_modele(F, [i + 1 if b else -(i + 1) for i, b in enumerate(bits)])
              for bits in itertools.product((0, 1), repeat=n))
    formules.append((n, F, ref))

for nom, h in HEURISTIQUES.items():
    for n, F, ref in formules:
        sat, v = dpll_h(F, h)
        assert sat == ref and (not sat or verifier_modele(F, completer(v, n))), (nom, F)
    print(f"{nom} : accord avec la force brute sur les 500 formules")
nb_sat = sum(ref for _, _, ref in formules)
print(f"({nb_sat} SAT, {500 - nb_sat} UNSAT)")

H1 : accord avec la force brute sur les 500 formules
H2 : accord avec la force brute sur les 500 formules
H3 : accord avec la force brute sur les 500 formules
H4 : accord avec la force brute sur les 500 formules
(147 SAT, 353 UNSAT)


**b.** Instances des enseignants, comme à la question 2 (processus séparé `python heuristiques.py --heur H`, limite de 60 s) : le statut est comparé à `expected.json` et les modèles sont vérifiés.

In [28]:
def executer_h(nom, heur):
    chemin = f"../../public-instances/instances/{nom}"
    t0 = time.perf_counter()
    try:
        r = subprocess.run([sys.executable, "heuristiques.py", chemin, "--heur", heur],
                           capture_output=True, text=True, timeout=LIMITE)
    except subprocess.TimeoutExpired:
        return "TIMEOUT", None, LIMITE
    duree = time.perf_counter() - t0
    if r.returncode != 0:
        return "ERREUR", False, duree
    sortie = r.stdout.splitlines()
    statut = "SAT" if "s SATISFIABLE" in sortie else "UNSAT"
    correct = statut == attendu[nom]
    if correct and statut == "SAT":
        modele = [int(x) for x in sortie[1].split()[1:-1]]
        correct = verifier_modele(lire_dimacs(chemin)[1], modele)
    return statut, correct, duree

def cellule(statut, correct, duree):
    return "timeout" if statut == "TIMEOUT" else f"{'ok' if correct else 'FAUX'} {duree:6.2f}s"

taches = [(nom, h) for nom in sorted(attendu) for h in HEURISTIQUES]
with ThreadPoolExecutor(4) as ex:
    res_h = dict(zip(taches, ex.map(lambda t: executer_h(*t), taches)))

print(f"{'instance':<34}{'attendu':>8}" + "".join(f"{h:>14}" for h in HEURISTIQUES))
for nom in sorted(attendu):
    print(f"{nom:<34}{attendu[nom]:>8}" + "".join(f"{cellule(*res_h[nom, h]):>14}" for h in HEURISTIQUES))
print()
for h in HEURISTIQUES:
    r = [res_h[nom, h] for nom in attendu]
    print(f"{h} : {sum(c is True for _, c, _ in r)}/{len(r)} correctes, "
          f"{sum(c is False for _, c, _ in r)} fausses, {sum(s == 'TIMEOUT' for s, _, _ in r)} timeouts")

instance                           attendu            H1            H2            H3            H4
cli_model_sat.cnf                      SAT    ok   0.06s    ok   0.06s    ok   0.06s    ok   0.06s
edge_all_negative_sat.cnf              SAT    ok   0.04s    ok   0.04s    ok   0.04s    ok   0.04s
edge_duplicate_literals_unsat.cnf    UNSAT    ok   0.05s    ok   0.04s    ok   0.04s    ok   0.04s
edge_empty_clause_unsat.cnf          UNSAT    ok   0.05s    ok   0.05s    ok   0.05s    ok   0.05s
edge_empty_formula_sat.cnf             SAT    ok   0.05s    ok   0.05s    ok   0.05s    ok   0.05s
edge_no_clause_sat.cnf                 SAT    ok   0.06s    ok   0.06s    ok   0.05s    ok   0.05s
edge_tautology_sat.cnf                 SAT    ok   0.06s    ok   0.06s    ok   0.06s    ok   0.05s
public-01.cnf                          SAT    ok   0.06s    ok   0.05s    ok   0.05s    ok   0.05s
public-02.cnf                          SAT    ok   0.05s    ok   0.05s    ok   0.05s    ok   0.05s
public-03.

**c.** Les 51 grilles Meowdoku de la question 1, avec la limite de 60 s : le fichier DIMACS de chaque grille est résolu par `python heuristiques.py`, puis la valuation est décodée en chats et contrôlée par `verifier_solution`.

In [29]:
def resoudre_grille_h(nom, heur):
    t0 = time.perf_counter()
    try:
        r = subprocess.run([sys.executable, "heuristiques.py", f"cnf/{nom}.cnf", "--heur", heur],
                           capture_output=True, text=True, timeout=LIMITE)
    except subprocess.TimeoutExpired:
        return None
    duree = time.perf_counter() - t0
    sortie = r.stdout.splitlines()
    if r.returncode != 0 or sortie[0] != "s SATISFIABLE":
        raise RuntimeError(f"{nom}, {heur} : {r.stdout}{r.stderr}")
    g = lire_grille(f"grilles/{nom}.txt")
    chats = chats_depuis_valuation({x for x in map(int, sortie[1].split()[1:-1]) if x > 0}, g.M)
    if verifier_solution(g, chats):
        raise RuntimeError(f"{nom}, {heur} : solution illicite")
    return duree

taches = [(nom, h) for nom in noms for h in HEURISTIQUES]
with ThreadPoolExecutor(4) as ex:
    durees_h = dict(zip(taches, ex.map(lambda t: resoudre_grille_h(*t), taches)))

print(f"{'grille':<12}" + "".join(f"{h:>10}" for h in HEURISTIQUES))
for nom in noms:
    print(f"{nom:<12}" + "".join("   timeout" if durees_h[nom, h] is None else f"{durees_h[nom, h]:9.2f}s"
                                 for h in HEURISTIQUES))
print()
for h in HEURISTIQUES:
    d = [durees_h[nom, h] for nom in noms]
    print(f"{h} : {sum(x is not None for x in d)}/51 grilles résolues en moins de {LIMITE} s, "
          "toutes les solutions sont vérifiées")

grille              H1        H2        H3        H4
meow_04_0        0.08s     0.08s     0.08s     0.07s
meow_04_1        0.05s     0.05s     0.04s     0.04s
meow_04_2        0.05s     0.05s     0.05s     0.05s
meow_05_0        0.05s     0.04s     0.04s     0.04s
meow_05_1        0.05s     0.05s     0.04s     0.05s
meow_05_2        0.05s     0.04s     0.04s     0.04s
meow_06_0        0.05s     0.05s     0.05s     0.05s
meow_06_1        0.04s     0.05s     0.05s     0.05s
meow_06_2        0.05s     0.05s     0.05s     0.05s
meow_07_0        0.05s     0.05s     0.05s     0.05s
meow_07_1        0.06s     0.06s     0.06s     0.06s
meow_07_2        0.05s     0.05s     0.05s     0.05s
meow_08_0        0.06s     0.06s     0.06s     0.06s
meow_08_1        0.05s     0.05s     0.05s     0.05s
meow_08_2        0.06s     0.05s     0.05s     0.06s
meow_09_0        0.06s     0.07s     0.07s     0.07s
meow_09_1        0.06s     0.07s     0.07s     0.07s
meow_09_2        0.06s     0.07s     0.07s    